<a href="https://colab.research.google.com/github/anupbagale/python-for-ai-ml/blob/main/07-pandas/01_lesson.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 7: pandas

In Module 5, loading the penguin data, converting types, handling missing values, splitting and saving took a page of code built on `csv` and dictionaries. In Module 6, NumPy made the number-crunching fast, but NumPy arrays have no column names and hold one type each, so a table that mixes species names with measurements doesn't fit naturally.

**pandas** fills that gap. It gives you the **DataFrame**: a table with named columns, where each column is a NumPy array with its own type, and rows have labels too. It's the standard tool for loading, cleaning, exploring and reshaping tabular data in Python, and the step before almost every classical ML model.

This module covers the parts of pandas you'll use daily, and the traps that make pandas code silently wrong. Because you built all of this by hand in Module 5, you'll be able to see exactly what each pandas function does for you.

**Contents**

1. Series and DataFrames
2. Reading data
3. Selecting rows and columns
4. Copies, views, and modifying data safely
5. Missing values
6. Creating and transforming columns
7. groupby: split, apply, combine
8. Combining tables with merge and concat
9. Categories and one-hot encoding
10. Performance: vectorize, don't loop
11. Putting it together: the Module 5 pipeline in pandas

The code here works on pandas 2.x and 3.x. Where the two versions display something differently, the text points it out.

## 1. Series and DataFrames

A **Series** is a one-dimensional array with labels (an *index*):

In [1]:
import numpy as np
import pandas as pd

print("pandas", pd.__version__)

mass = pd.Series([3750.0, 3800.0, 3250.0], index=["p1", "p2", "p3"], name="body_mass_g")
print(mass)
print(mass["p2"])
print(mass.values)      # the underlying NumPy array

pandas 3.0.5
p1    3750.0
p2    3800.0
p3    3250.0
Name: body_mass_g, dtype: float64
3800.0
[3750. 3800. 3250.]


A **DataFrame** is a collection of Series that share the same index: a table.

In [2]:
df = pd.DataFrame({
    "species": ["Adelie", "Adelie", "Gentoo"],
    "bill_length_mm": [39.1, 39.5, 46.1],
    "body_mass_g": [3750, 3800, 4500],
})
print(df)
print()
print(df.dtypes)
print(df.shape)

  species  bill_length_mm  body_mass_g
0  Adelie            39.1         3750
1  Adelie            39.5         3800
2  Gentoo            46.1         4500

species               str
bill_length_mm    float64
body_mass_g         int64
dtype: object
(3, 3)


Each column has its own dtype. The numbers are NumPy `float64` and `int64`. The species column holds text. In pandas 3 it shows as `str`; in older versions it shows as `object`, which means "Python objects", usually strings.

The numbers on the left (`0`, `1`, `2`) are the **index**: row labels. By default it's just positions, but it can be anything, like dates or IDs. Most operations line rows up by index label, which is powerful and occasionally surprising, as you'll see.

## 2. Reading data

Remember the penguin loader from Module 5? Reading the file, recognising `NA` as missing and converting every column to the right type. In pandas, it's one line:

In [3]:
from pathlib import Path

DATA_URL = "https://raw.githubusercontent.com/anupbagale/python-for-ai-ml/main/05-files-and-data/data/penguins.csv"
local = Path("data/penguins.csv")

penguins = pd.read_csv(local if local.exists() else DATA_URL)
print(penguins.shape)
penguins.head()

(344, 8)


,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex,year
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,male,2007
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,female,2007
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,female,2007
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN,2007
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,female,2007


`read_csv` accepts a path or a URL. It detected the column types, and turned the `NA` entries into `NaN` (pandas' missing-value marker for numbers). The last expression in a cell, like `penguins.head()`, is displayed as a formatted table in Colab and Jupyter.

A few first looks you'll run on every new dataset:

In [4]:
penguins.info()

<class 'pandas.DataFrame'>
RangeIndex: 344 entries, 0 to 343
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   species            344 non-null    str    
 1   island             344 non-null    str    
 2   bill_length_mm     342 non-null    float64
 3   bill_depth_mm      342 non-null    float64
 4   flipper_length_mm  342 non-null    float64
 5   body_mass_g        342 non-null    float64
 6   sex                333 non-null    str    
 7   year               344 non-null    int64  
dtypes: float64(4), int64(1), str(3)
memory usage: 21.6 KB


`info()` shows each column's type and how many non-missing values it has: 344 rows, but only 342 values for the measurements and 333 for `sex`. That's the missing-value report from Module 5, for free.

In [5]:
penguins.describe()

,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,year
count,342.000000,342.000000,342.000000,342.000000,344.000000
mean,43.921930,17.151170,200.915205,4201.754386,2008.029070
std,5.459584,1.974793,14.061714,801.954536,0.818356
min,32.100000,13.100000,172.000000,2700.000000,2007.000000
25%,39.225000,15.600000,190.000000,3550.000000,2007.000000
50%,44.450000,17.300000,197.000000,4050.000000,2008.000000
75%,48.500000,18.700000,213.000000,4750.000000,2009.000000
max,59.600000,21.500000,231.000000,6300.000000,2009.000000


`describe()` gives summary statistics for each numeric column. Skim it for anything implausible: negative masses, a maximum of 9999 or a standard deviation of zero all point to data problems.

In [6]:
print(penguins["species"].value_counts())
print()
print(penguins["species"].value_counts(normalize=True).round(3))

species
Adelie       152
Gentoo       124
Chinstrap     68
Name: count, dtype: int64

species
Adelie       0.442
Gentoo       0.360
Chinstrap    0.198
Name: proportion, dtype: float64


### Telling read_csv what it can't guess

`read_csv` guesses, and guesses can be wrong. The main arguments that fix it:

- `na_values=["?", "-"]`: extra strings that mean "missing" (it already knows `NA`, `NaN`, `null` and the empty string)
- `dtype={"zip_code": str}`: force a type, for example to keep leading zeros in codes
- `parse_dates=["date"]`: turn a column into real dates
- `encoding="utf-8-sig"`: the Excel byte order mark from Module 5
- `usecols=[...]`, `nrows=1000`: read only part of a big file

The leading-zeros problem is worth seeing once:

In [7]:
from io import StringIO

text = "customer,zip_code\nA,01234\nB,98765\n"
print(pd.read_csv(StringIO(text)))
print(pd.read_csv(StringIO(text), dtype={"zip_code": str}))

  customer  zip_code
0        A      1234
1        B     98765
  customer zip_code
0        A    01234
1        B    98765


`StringIO` makes a string behave like a file, handy for small examples. By default, `01234` became the number `1234`. If that column is later joined to another table where the code is stored as text, nothing matches, and you get a mysteriously empty result.

## 3. Selecting rows and columns

### Columns

In [8]:
print(penguins["species"].head(3))                        # one column -> Series
print(penguins[["species", "body_mass_g"]].head(3))       # a list of columns -> DataFrame

0    Adelie
1    Adelie
2    Adelie
Name: species, dtype: str
  species  body_mass_g
0  Adelie       3750.0
1  Adelie       3800.0
2  Adelie       3250.0


Note the double brackets for several columns: the outer pair is indexing, the inner pair is a Python list.

### Rows: loc and iloc

There are two ways to select rows, and mixing them up is a common source of bugs:

- **`.loc[...]`** selects by **label**: index labels and column names.
- **`.iloc[...]`** selects by **position**: 0, 1, 2, like a NumPy array.

In [9]:
print(penguins.loc[0, "species"])                 # row label 0, column "species"
print(penguins.loc[0:2, ["species", "island"]])   # labels 0 to 2, INCLUDING 2
print(penguins.iloc[0:2, 0:2])                    # positions 0 and 1, like normal slicing

Adelie
  species     island
0  Adelie  Torgersen
1  Adelie  Torgersen
2  Adelie  Torgersen
  species     island
0  Adelie  Torgersen
1  Adelie  Torgersen


Watch the slice ends: `.loc[0:2]` **includes** label 2, while `.iloc[0:2]` excludes position 2, like every other slice in Python. With the default numeric index, labels and positions start out the same, but after filtering or sorting they don't, which is when `loc` and `iloc` start giving different answers:

In [10]:
gentoo = penguins[penguins["species"] == "Gentoo"]
print(gentoo.index[:5].tolist())
print(gentoo.iloc[0]["island"], "| row label of the first Gentoo:", gentoo.index[0])

try:
    gentoo.loc[0]
except KeyError:
    print("gentoo.loc[0] raises KeyError: there's no row labelled 0 in this subset")

[152, 153, 154, 155, 156]
Biscoe | row label of the first Gentoo: 152
gentoo.loc[0] raises KeyError: there's no row labelled 0 in this subset


### Filtering with boolean masks

Filtering works exactly like NumPy masks from Module 6, including `&`, `|`, `~` and the parentheses:

In [11]:
heavy_gentoo = penguins[(penguins["species"] == "Gentoo") & (penguins["body_mass_g"] > 5500)]
print(len(heavy_gentoo))

on_two_islands = penguins[penguins["island"].isin(["Dream", "Biscoe"])]
print(on_two_islands["island"].value_counts())

28
island
Biscoe    168
Dream     124
Name: count, dtype: int64


`.query()` lets you write the same thing as a string, which some people find easier to read:

In [12]:
print(len(penguins.query("species == 'Gentoo' and body_mass_g > 5500")))

28


## 4. Copies, views, and modifying data safely

This is the most confusing part of pandas, so it's worth slowing down.

### The rule

When you want to change values in a DataFrame, do it **in one step with `.loc`**, using a row selection and a column name together:

In [13]:
df = penguins.copy()
df.loc[df["body_mass_g"] > 6000, "size_class"] = "very large"
df.loc[df["body_mass_g"] <= 6000, "size_class"] = "normal"
print(df["size_class"].value_counts(dropna=False))

size_class
normal        340
NaN             2
very large      2
Name: count, dtype: int64


Look at the result: two penguins got `NaN`, not either label. They're the two with no recorded body mass, and `NaN > 6000` and `NaN <= 6000` are both `False` (the NaN comparison rule from Module 1), so neither assignment touched them. Missing values fall through conditions silently. When you split data into categories with conditions, it's worth checking that the categories add up to the total.

### What not to do

The trap is *chained* indexing, where you select in two steps and then assign:

```python
df[df["body_mass_g"] > 6000]["size_class"] = "very large"     # don't
```

The first step, `df[mask]`, produces a new object, and the assignment then changes *that* object, not `df`. Your data is left unchanged. Depending on the pandas version you either get a warning (`SettingWithCopyWarning` in pandas 2, `ChainedAssignmentError` in pandas 3) or the change just quietly doesn't happen. Either way, the fix is always the single `.loc[rows, column] = value` form above.

### Subsets you plan to modify

The same issue appears in a less obvious form. You filter a subset, keep it in a variable, and later change it:

In [14]:
adelie = penguins[penguins["species"] == "Adelie"].copy()
adelie["mass_kg"] = adelie["body_mass_g"] / 1000

print("mass_kg" in adelie.columns, "mass_kg" in penguins.columns)

True False


The explicit `.copy()` says "this is a separate table now". It avoids any warning in older versions, and it makes the intent obvious to anyone reading the code. It's the same idea as `.copy()` for NumPy views in Module 6. A good habit: whenever you create a subset that you're going to add columns to or modify, end the line with `.copy()`.

pandas 3 made this simpler under the hood with *copy-on-write*: every derived DataFrame behaves like a copy, and changing it never affects the original. The habit above gives the same behaviour on every version.

## 5. Missing values

pandas uses `NaN` (and `None`, `NaT` for dates, or `pd.NA`) to mark missing values. The tools mirror what you wrote by hand in Module 5:

In [15]:
print(penguins.isna().sum())
print()
print("rows with any missing value:", penguins.isna().any(axis=1).sum())

species               0
island                0
bill_length_mm        2
bill_depth_mm         2
flipper_length_mm     2
body_mass_g           2
sex                  11
year                  0
dtype: int64

rows with any missing value: 11


`isna()` gives a DataFrame of `True`/`False`, and `sum()` counts the `True`s per column. `axis=1` works across columns, row by row, as in NumPy.

**Dropping:**

In [16]:
measurements = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]

complete = penguins.dropna(subset=measurements)
print(len(penguins), "->", len(complete))

344 -> 342


`subset=` matters. Plain `dropna()` drops a row if *any* column is missing, which here would also throw away the nine penguins whose only gap is `sex`, losing perfectly good measurements.

**Filling:**

In [17]:
filled = complete.fillna({"sex": "unknown"})
print(filled["sex"].value_counts())

sex
male       168
female     165
unknown      9
Name: count, dtype: int64


For numbers, the leakage rule from Modules 1 and 5 still applies: compute fill values from the **training data only**, then use them for both sets. `fillna` makes that easy, since you pass the values in:

```python
medians = train[numeric_columns].median()
train = train.fillna(medians)
test = test.fillna(medians)          # the training medians, not the test medians
```

**Aggregations skip missing values by default:**

In [18]:
print(penguins["body_mass_g"].mean())              # NaN values are ignored
print(penguins["body_mass_g"].to_numpy().mean())   # NumPy doesn't ignore them

4201.754385964912
nan


That's the difference flagged in Module 5: pandas' `mean` skips missing values, NumPy's doesn't. Neither is wrong, but you should know which one you're calling.

## 6. Creating and transforming columns

### Arithmetic is vectorized

Column operations work on whole columns at once, like NumPy arrays:

In [19]:
df = complete.copy()
df["bill_ratio"] = df["bill_length_mm"] / df["bill_depth_mm"]
df["mass_kg"] = df["body_mass_g"] / 1000
print(df[["bill_length_mm", "bill_depth_mm", "bill_ratio", "mass_kg"]].head(3))

   bill_length_mm  bill_depth_mm  bill_ratio  mass_kg
0            39.1           18.7    2.090909     3.75
1            39.5           17.4    2.270115     3.80
2            40.3           18.0    2.238889     3.25


`assign` does the same but returns a new DataFrame, leaving the original untouched, which suits step-by-step pipelines:

In [20]:
summary = (
    complete
    .assign(mass_kg=lambda d: d["body_mass_g"] / 1000)
    .query("mass_kg > 5")
    [["species", "island", "mass_kg"]]
)
print(summary.head())

    species  island  mass_kg
153  Gentoo  Biscoe     5.70
155  Gentoo  Biscoe     5.70
156  Gentoo  Biscoe     5.40
159  Gentoo  Biscoe     5.20
161  Gentoo  Biscoe     5.15


Wrapping the chain in parentheses lets you put each step on its own line. The `lambda d:` receives the DataFrame as it is at that step of the chain.

### Conditions

`np.where` from Module 6 works on columns:

In [21]:
df["heavy"] = np.where(df["body_mass_g"] > 4500, "heavy", "light")
print(df["heavy"].value_counts())

heavy
light    227
heavy    115
Name: count, dtype: int64


### Mapping values

`map` with a dictionary replaces each value with its mapping, the same idea as the cleaning dictionaries in Module 5:

In [22]:
species_code = {"Adelie": 0, "Chinstrap": 1, "Gentoo": 2}
df["species_id"] = df["species"].map(species_code)
print(df[["species", "species_id"]].drop_duplicates())

       species  species_id
0       Adelie           0
152     Gentoo           2
276  Chinstrap           1


Anything not in the dictionary becomes `NaN`. That's a feature: a typo like `"gentoo"` shows up as a missing value you'll notice, rather than being guessed.

### String methods

The `.str` accessor applies string methods to a whole column, which is how messy text gets cleaned:

In [23]:
messy = pd.Series([" Adelie ", "ADELIE", "Gentoo penguin", "chinstrap", None])
clean = messy.str.strip().str.lower().str.replace(" penguin", "", regex=False).str.title()
print(clean.tolist())

['Adelie', 'Adelie', 'Gentoo', 'Chinstrap', nan]


Missing values pass straight through as missing, without raising errors.

### Dates

Converting text to real dates gives you date arithmetic and the `.dt` accessor:

In [24]:
events = pd.DataFrame({"when": ["2025-01-15", "2025-02-03", "2025-03-28"], "amount": [20.0, 35.5, 12.0]})
events["when"] = pd.to_datetime(events["when"])

events["month"] = events["when"].dt.month
events["weekday"] = events["when"].dt.day_name()
events["days_since_first"] = (events["when"] - events["when"].min()).dt.days
print(events)

        when  amount  month    weekday  days_since_first
0 2025-01-15    20.0      1  Wednesday                 0
1 2025-02-03    35.5      2     Monday                19
2 2025-03-28    12.0      3     Friday                72


Subtracting two dates gives a duration (a `Timedelta`), and `.dt.days` turns it into a plain number of days, a typical feature for ML. The mini project leans on this heavily.

## 7. groupby: split, apply, combine

`groupby` is the heart of pandas. It **splits** the rows into groups, **applies** a calculation to each group, and **combines** the results into a new table. You wrote this by hand in Module 2 with `defaultdict(list)`, and the per-species table in Module 5 took a dozen lines.

In [25]:
print(complete.groupby("species")["body_mass_g"].mean())

species
Adelie       3700.662252
Chinstrap    3733.088235
Gentoo       5076.016260
Name: body_mass_g, dtype: float64


Several statistics at once, with **named aggregation**, where each output column is `name=(input_column, function)`:

In [26]:
by_species = complete.groupby("species").agg(
    n=("body_mass_g", "size"),
    mean_mass=("body_mass_g", "mean"),
    max_flipper=("flipper_length_mm", "max"),
    mean_bill=("bill_length_mm", "mean"),
).round(1)
print(by_species)

             n  mean_mass  max_flipper  mean_bill
species                                          
Adelie     151     3700.7        210.0       38.8
Chinstrap   68     3733.1        212.0       48.8
Gentoo     123     5076.0        231.0       47.5


Grouping by two columns gives one row per combination:

In [27]:
print(complete.groupby(["species", "island"]).size())

species    island   
Adelie     Biscoe        44
           Dream         56
           Torgersen     51
Chinstrap  Dream         68
Gentoo     Biscoe       123
dtype: int64


That reveals something we noted in Module 5: Gentoos were only recorded on Biscoe, and Chinstraps only on Dream. `size()` counts rows per group.

### transform: one value per row

`agg` gives one row per *group*. `transform` gives one value per *original row*, which is what you want when the group statistic becomes a new feature. For example, each penguin's mass compared to the average for its species:

In [28]:
df = complete.copy()
df["species_mean_mass"] = df.groupby("species")["body_mass_g"].transform("mean")
df["mass_vs_species"] = df["body_mass_g"] - df["species_mean_mass"]
print(df[["species", "body_mass_g", "species_mean_mass", "mass_vs_species"]].head())

  species  body_mass_g  species_mean_mass  mass_vs_species
0  Adelie       3750.0        3700.662252        49.337748
1  Adelie       3800.0        3700.662252        99.337748
2  Adelie       3250.0        3700.662252      -450.662252
4  Adelie       3450.0        3700.662252      -250.662252
5  Adelie       3650.0        3700.662252       -50.662252


### crosstab and pivot_table

For a quick table of counts between two columns, `crosstab` is the shortcut:

In [29]:
print(pd.crosstab(complete["species"], complete["island"]))

island     Biscoe  Dream  Torgersen
species                            
Adelie         44     56         51
Chinstrap       0     68          0
Gentoo        123      0          0


`pivot_table` generalises it to any aggregation:

In [30]:
print(complete.pivot_table(index="species", columns="sex", values="body_mass_g", aggfunc="mean").round(0))

sex        female    male
species                  
Adelie     3369.0  4043.0
Chinstrap  3527.0  3939.0
Gentoo     4680.0  5485.0


Males are heavier in every species. A feature like `sex` could help a model that predicts body mass.

## 8. Combining tables

Real data usually comes in several tables that have to be joined: customers in one, orders in another, products in a third.

In [31]:
orders = pd.DataFrame({
    "order_id": [1, 2, 3, 4, 5],
    "customer_id": ["c1", "c2", "c1", "c3", "c9"],
    "amount": [20.0, 35.0, 12.5, 50.0, 8.0],
})
customers = pd.DataFrame({
    "customer_id": ["c1", "c2", "c3", "c4"],
    "country": ["Nepal", "India", "Nepal", "Japan"],
})

joined = orders.merge(customers, on="customer_id", how="left")
print(joined)

   order_id customer_id  amount country
0         1          c1    20.0   Nepal
1         2          c2    35.0   India
2         3          c1    12.5   Nepal
3         4          c3    50.0   Nepal
4         5          c9     8.0     NaN


`how="left"` keeps every order, even ones whose customer isn't in the customers table: order 5's `c9` gets `NaN` for country. The other options:

| `how` | keeps |
|---|---|
| `"left"` | every row of the left table |
| `"inner"` | only rows with a match in both (the default) |
| `"right"` | every row of the right table |
| `"outer"` | every row of both |

The default, `inner`, silently **drops** unmatched rows. Losing order 5 without noticing could be a real problem, so it's worth being explicit about `how` every time.

`indicator=True` adds a column saying where each row came from, which is a quick way to audit a join:

In [32]:
print(orders.merge(customers, on="customer_id", how="outer", indicator=True)[["customer_id", "_merge"]])

  customer_id      _merge
0          c1        both
1          c1        both
2          c2        both
3          c3        both
4          c4  right_only
5          c9   left_only


### The duplicate key trap

If the key isn't unique on the side you expect it to be unique, a join **multiplies rows**:

In [33]:
customers_dup = pd.concat([customers, pd.DataFrame({"customer_id": ["c1"], "country": ["Nepal"]})])
bad = orders.merge(customers_dup, on="customer_id", how="left")
print(len(orders), "orders ->", len(bad), "rows after the join")

5 orders -> 7 rows after the join


Customer `c1` appears twice in the customer table, so each of their orders appears twice after the join, and total revenue would be overcounted. This happens a lot with real data. The `validate` argument makes pandas check the relationship and fail loudly instead:

In [34]:
try:
    orders.merge(customers_dup, on="customer_id", how="left", validate="many_to_one")
except pd.errors.MergeError as e:
    print("MergeError:", e)

MergeError: Merge keys are not unique in right dataset; not a many-to-one merge

Duplicates in right:
 customer_id
         c1 ...


`validate="many_to_one"` says "many orders per customer, but each customer exactly once on the right". Use it whenever you join a table of events to a table of entities.

### concat: stacking tables

In [35]:
jan = pd.DataFrame({"day": [1, 2], "sales": [100, 120]})
feb = pd.DataFrame({"day": [1, 2], "sales": [90, 140]})
both = pd.concat([jan.assign(month="Jan"), feb.assign(month="Feb")], ignore_index=True)
print(both)

   day  sales month
0    1    100   Jan
1    2    120   Jan
2    1     90   Feb
3    2    140   Feb


`ignore_index=True` renumbers the rows. Without it, both tables keep their own index, giving duplicate labels `0, 1, 0, 1`, which makes `.loc` behave in confusing ways later.

## 9. Categories and one-hot encoding

Models need numbers. For a categorical column like `island`, the standard approach is **one-hot encoding** (as in Modules 1 and 6): one 0/1 column per category. `pd.get_dummies` does it for a DataFrame:

In [36]:
encoded = pd.get_dummies(complete[["island", "bill_length_mm"]], columns=["island"], dtype=int)
print(encoded.head())

   bill_length_mm  island_Biscoe  island_Dream  island_Torgersen
0            39.1              0             0                 1
1            39.5              0             0                 1
2            40.3              0             0                 1
4            36.7              0             0                 1
5            39.3              0             0                 1


### The train/test mismatch trap

If you encode the training and test sets separately, and a category is missing from one of them, they end up with **different columns**:

In [37]:
train_part = complete[complete["island"] != "Torgersen"]
test_part = complete[complete["island"] == "Torgersen"]

train_enc = pd.get_dummies(train_part[["island"]], dtype=int)
test_enc = pd.get_dummies(test_part[["island"]], dtype=int)
print(train_enc.columns.tolist())
print(test_enc.columns.tolist())

['island_Biscoe', 'island_Dream']
['island_Torgersen']


A model trained on the first set of columns can't be applied to the second. The fix is to treat the **training columns as the definition**, and reindex the test set to match: missing columns are filled with 0, and categories the model has never seen are dropped.

In [38]:
test_aligned = test_enc.reindex(columns=train_enc.columns, fill_value=0)
print(test_aligned.head(3))

   island_Biscoe  island_Dream
0              0             0
1              0             0
2              0             0


This is the "fit on train, apply to test" rule once again, for encoding this time. scikit-learn's `OneHotEncoder` stores the categories it saw during `fit` for exactly this reason.

### The category dtype

For a column with a few repeated values, the `category` dtype stores each distinct value once and uses small integer codes underneath. It saves memory on big tables and makes the codes easy to get at:

In [39]:
species_cat = complete["species"].astype("category")
print(species_cat.cat.categories.tolist())
print(species_cat.cat.codes.head().tolist())
print(complete["species"].memory_usage(deep=True), "bytes as text vs", species_cat.memory_usage(deep=True), "as category")

['Adelie', 'Chinstrap', 'Gentoo']
[0, 0, 0, 0, 0]
21750 bytes as text vs 3246 as category


## 10. Performance: vectorize, don't loop

pandas is built on NumPy, so the Module 6 lesson applies: whole-column operations are fast, and Python loops over rows are slow. There are three ways to compute the same new column:

In [40]:
import time

big = pd.concat([complete] * 300, ignore_index=True)       # about 100,000 rows
print(len(big), "rows")

start = time.perf_counter()
ratios = [row["bill_length_mm"] / row["bill_depth_mm"] for _, row in big.iterrows()]
t_iter = time.perf_counter() - start

start = time.perf_counter()
ratios = big.apply(lambda row: row["bill_length_mm"] / row["bill_depth_mm"], axis=1)
t_apply = time.perf_counter() - start

start = time.perf_counter()
ratios = big["bill_length_mm"] / big["bill_depth_mm"]
t_vec = time.perf_counter() - start

print(f"iterrows:   {t_iter:.3f} s")
print(f"apply:      {t_apply:.3f} s")
print(f"vectorized: {t_vec:.5f} s   ({t_iter / t_vec:,.0f}x faster than iterrows)")

102600 rows


iterrows:   2.435 s
apply:      0.534 s
vectorized: 0.00068 s   (3,597x faster than iterrows)


`iterrows` builds a whole Series object for every row. `apply(..., axis=1)` is a bit better, but it still calls a Python function per row. The vectorized version is a single NumPy operation. Before reaching for `apply`, check whether arithmetic, `np.where`, `.map`, `.str` or `.dt` can do it, because they almost always can.

## 11. Putting it together: the Module 5 pipeline in pandas

In Module 5, the penguin pipeline loaded the raw CSV, dropped rows with no measurements, did a stratified split with a seed, filled missing `sex` values, wrote train and test files with a metadata record, and read everything back to check it. Here's the same pipeline in pandas:

In [41]:
import json

SEED = 42
work = Path("work") / "processed"
work.mkdir(parents=True, exist_ok=True)

raw = pd.read_csv(local if local.exists() else DATA_URL)
usable = raw.dropna(subset=measurements).fillna({"sex": "unknown"})

test = usable.groupby("species").sample(frac=0.2, random_state=SEED)
train = usable.drop(index=test.index)

train.to_csv(work / "train.csv", index=False)
test.to_csv(work / "test.csv", index=False)

metadata = {
    "rows_raw": len(raw),
    "rows_dropped": len(raw) - len(usable),
    "rows_train": len(train),
    "rows_test": len(test),
    "species_test": test["species"].value_counts().sort_index().to_dict(),
}
(work / "metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
print(json.dumps(metadata, indent=2))

{
  "rows_raw": 344,
  "rows_dropped": 2,
  "rows_train": 273,
  "rows_test": 69,
  "species_test": {
    "Adelie": 30,
    "Chinstrap": 14,
    "Gentoo": 25
  }
}


The steps line up one to one with Module 5:

- `dropna(subset=...)` is the "drop rows with no measurements" loop.
- `fillna({"sex": "unknown"})` is the fill loop.
- `groupby("species").sample(frac=0.2, random_state=SEED)` is the whole `stratified_split` function: take 20% of each species, reproducibly.
- `drop(index=test.index)` gives the training set: every row whose index label isn't in the test set. This is where a meaningful index pays off.
- `to_csv(..., index=False)` is the `DictWriter` code. `index=False` stops pandas writing the row labels as an extra, unnamed first column, which is a common nuisance when the file is read back.

The counts match Module 5 exactly: 344 raw rows, 2 dropped, and 273 training and 69 test rows with the same number of each species. The specific penguins in each set differ, because pandas shuffles differently from our hand-written function. And the same read-back checks:

In [42]:
train_back = pd.read_csv(work / "train.csv")
test_back = pd.read_csv(work / "test.csv")

assert len(train_back) == metadata["rows_train"] and len(test_back) == metadata["rows_test"]
assert train_back[measurements].notna().all().all()
assert not set(train.index) & set(test.index)

overlap = train_back.merge(test_back, how="inner")
print("rows that appear in both files:", len(overlap))
print("all checks passed")

rows that appear in both files: 0
all checks passed


`merge(..., how="inner")` with no `on=` joins on every shared column, so it finds rows that are identical in both files, the leakage check from Module 5 in one line. If it finds any, it's worth investigating: here it would mean two different penguins with exactly the same measurements, sex and year, which is possible but rare.

Twenty lines instead of a page, and each one says what it does. That's why pandas is the default for tabular data, but you now know what's underneath each step, which is what lets you spot when it guesses wrong.

## Summary

- A **DataFrame** is a table of named columns, each a NumPy array with its own dtype, sharing a row **index**.
- `read_csv` handles parsing and missing values. Tell it what it can't guess: `na_values`, `dtype`, `parse_dates`, `encoding`. First looks: `head`, `info`, `describe`, `value_counts`.
- `.loc` selects by label (slices include the end), `.iloc` by position. After filtering, labels and positions differ.
- Modify with a single `df.loc[rows, col] = value`. Never chain `df[mask][col] = value`. End a subset you'll modify with `.copy()`.
- `isna().sum()`, `dropna(subset=...)`, `fillna(values_from_train)`. pandas skips `NaN` in aggregations; NumPy doesn't.
- Transform columns with arithmetic, `assign`, `np.where`, `map`, `.str` and `.dt`.
- `groupby(...).agg(name=(col, func))` gives one row per group; `transform` gives one value per row.
- `merge` with an explicit `how`, and `validate="many_to_one"` to catch duplicate keys. `concat(..., ignore_index=True)` to stack.
- One-hot encode with `get_dummies`, then `reindex(columns=train_columns, fill_value=0)` for test data.
- Vectorize. `iterrows` and `apply(axis=1)` are last resorts.

Next, open `02_exercises.ipynb`.